In [5]:
from astropy.io import fits
from pathlib import Path
import numpy as np
import pandas as pd
from astropy.wcs import WCS
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
from sunpy.coordinates import frames, get_earth

In [6]:
# 1. Point this to your new folder
raw_dir = Path("sc_and_ph")

# 2. Name your output inventory file
out_csv = Path("fermi_file_inventory.csv")

# Create an empty list to hold our file data
rows = []
print("Cracking open Fermi FITS headers...")

# 3. Loop through every file in your folder
for fp in sorted(raw_dir.rglob("*")):
    # Skip it if it's a hidden folder or not a FITS file
    if not fp.is_file():
        continue
    if not (fp.name.lower().endswith(".fits") or fp.name.lower().endswith("fits.gz")):
        continue

    # 4. Setup the dictionary to record the file's info
    record = {
        "filename": fp.name,
        "filepath": str(fp), 
        "filesize_bytes": fp.stat().st_size, 
        "date_obs": None, 
        "date_end": None, 
        "ra": None, 
        "dec": None, 
        "instrument": None,
        "telescope": None, 
    }

    # 5. Pry open the FITS header and extract the date
    try:
        with fits.open(fp) as hdul:
            header = hdul[0].header
            record["date_obs"] = header.get("DATE-OBS")
            record["date_end"] = header.get("DATE-END")
            record["ra"] = header.get("RA")
            record["dec"] = header.get("DEC")
            record["instrument"] = header.get("INSTRUME")
            record["telescope"] = header.get("TELESCOP")
    except Exception as e:
        record["error"] = str(e)
    
    # Append this file's record to our giant list
    rows.append(record)

# 6. Convert the list to a Pandas DataFrame and save it as a CSV
df_raw = pd.DataFrame(rows)
df_raw.to_csv(out_csv, index=False)

print("\n--- INVENTORY COMPLETE ---")
print(f"Successfully logged {len(df_raw)} files to '{out_csv}'")

# Print just the filename and the date so we can fix the boopsie!
df_raw[['filename', 'date_obs']].head(18)

Cracking open Fermi FITS headers...

--- INVENTORY COMPLETE ---
Successfully logged 30 files to 'fermi_file_inventory.csv'


,filename,date_obs
0,L2603311255089B9590BF64_PH00.fits,2014-09-01T10:50:00.0000
1,L2603311255089B9590BF64_SC00.fits,2014-09-01T10:50:00.0000
2,L2604021201389C9590BF63_PH00.fits,2011-09-06T22:10:50.0000
3,L2604021201389C9590BF63_SC00.fits,2011-09-06T22:10:50.0000
4,L2604021204279C9590BF75_PH00.fits,2012-03-07T00:39:49.9999
5,L2604021204279C9590BF75_SC00.fits,2012-03-07T00:39:49.9999
6,L2604021212529C9590BF63_PH00.fits,2012-03-07T03:50:49.9999
7,L2604021212529C9590BF63_SC00.fits,2012-03-07T03:50:49.9999
8,L2604021230419C9590BF10_PH00.fits,2012-03-07T07:01:49.9999
9,L2604021230419C9590BF10_SC00.fits,2012-03-07T07:01:49.9999
